In [13]:
import pandas as pd
import numpy as np

from pathlib import Path

from backend.anomaly_detection.inference import predict_anomalies

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

print("Inference evaluation notebook initialized.")
print("Project root:", Path.cwd())

Inference evaluation notebook initialized.
Project root: f:\DataSets for Artificial Intelligence\greenko assignment\02_renewable_asset_fault_anomaly_copilot


In [14]:
import importlib

import backend.anomaly_detection.inference as inference_module

importlib.invalidate_caches()
importlib.reload(inference_module)

predict_anomalies = inference_module.predict_anomalies

print("Updated inference pipeline loaded.")
print("Loaded from:", inference_module.__file__)

Updated inference pipeline loaded.
Loaded from: f:\DataSets for Artificial Intelligence\greenko assignment\02_renewable_asset_fault_anomaly_copilot\backend\anomaly_detection\inference.py


In [15]:
# Locate the project data directory
data_dir = Path("data")

# Find available CSV files
csv_files = list(data_dir.rglob("*.csv"))

print("CSV files found:")

for file in csv_files:
    print(file)

CSV files found:
data\asset_telemetry.csv
data\engineered_data.csv


In [16]:
# Load raw telemetry dataset
data_path = Path("data") / "asset_telemetry.csv"

raw_df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Shape:", raw_df.shape)

display(raw_df.head())

Dataset loaded successfully.
Shape: (240, 8)


,timestamp_id,asset_id,hour,irradiance_wm2,wind_speed_ms,temperature_c,vibration_mm_s,power_mw
0,0,SOLAR_INV_01,0,0.0,6.57,26.26,1.37,0.0
1,1,SOLAR_INV_01,1,0.0,6.72,24.89,1.49,0.0
2,2,SOLAR_INV_01,2,0.0,6.26,23.85,1.25,0.0
3,3,SOLAR_INV_01,3,0.0,7.24,24.45,1.29,0.0
4,4,SOLAR_INV_01,4,0.0,7.39,26.45,1.50,0.0


In [17]:
# Dataset structure
print("Dataset shape:", raw_df.shape)

print("\nColumn names:")
print(raw_df.columns.tolist())

print("\nData types:")
display(raw_df.dtypes)

print("\nMissing values:")
display(raw_df.isnull().sum())

print("\nUnique assets:")
print(raw_df["asset_id"].nunique())

print("\nAsset distribution:")
display(raw_df["asset_id"].value_counts())

Dataset shape: (240, 8)

Column names:
['timestamp_id', 'asset_id', 'hour', 'irradiance_wm2', 'wind_speed_ms', 'temperature_c', 'vibration_mm_s', 'power_mw']

Data types:


timestamp_id        int64
asset_id           object
hour                int64
irradiance_wm2    float64
wind_speed_ms     float64
temperature_c     float64
vibration_mm_s    float64
power_mw          float64
dtype: object


Missing values:


timestamp_id      0
asset_id          0
hour              0
irradiance_wm2    0
wind_speed_ms     0
temperature_c     0
vibration_mm_s    0
power_mw          0
dtype: int64


Unique assets:
2

Asset distribution:


asset_id
SOLAR_INV_01    120
WTG_07          120
Name: count, dtype: int64

In [18]:
# Sort observations chronologically
raw_df = raw_df.sort_values(
    by=["timestamp_id", "asset_id"]
).reset_index(drop=True)

# 80% historical data, 20% incoming data
split_index = int(len(raw_df) * 0.80)

history_raw_df = raw_df.iloc[:split_index].copy()
incoming_raw_df = raw_df.iloc[split_index:].copy()

print("Historical observations:", len(history_raw_df))
print("Incoming observations:", len(incoming_raw_df))

Historical observations: 192
Incoming observations: 48


In [19]:
inference_output = predict_anomalies(
    history_df=history_raw_df,
    inference_df=incoming_raw_df,
    return_bundle=True
)

predictions_df = inference_output["predictions_df"]
anomalies_df = inference_output["anomalies_df"]
normal_df = inference_output["normal_df"]
anomaly_records = inference_output["anomaly_records"]

print("Inference completed successfully.")
print("Total predictions:", len(predictions_df))
print("Anomalies detected:", len(anomalies_df))
print("Normal observations:", len(normal_df))
print("RAG-ready records:", len(anomaly_records))

Data validation completed.
Data preprocessing completed.
Final dataset shape: (240, 8)
Inference completed successfully.
Total predictions: 48
Anomalies detected: 3
Normal observations: 45
RAG-ready records: 3


In [21]:
anomaly_records

[{'timestamp_id': 199,
  'asset_id': 'WTG_07',
  'irradiance_wm2': 0.0,
  'wind_speed_ms': 6.38,
  'temperature_c': 28.25,
  'vibration_mm_s': 1.21,
  'power_mw': 1.521,
  'asset_type': 'wind',
  'hour_sin': 0.9659258262890683,
  'hour_cos': -0.25881904510252063,
  'power_mw_rolling_median': 0.55,
  'power_mw_rolling_mad': 0.13249999999999998,
  'power_mw_median_deviation': 0.9709999999999999,
  'power_mw_modified_zscore': 4.94293962264151,
  'power_mw_zero_mad_deviation': 0,
  'vibration_mm_s_rolling_median': 1.355,
  'vibration_mm_s_rolling_mad': 0.09999999999999998,
  'vibration_mm_s_median_deviation': -0.14500000000000002,
  'vibration_mm_s_modified_zscore': -0.9780250000000004,
  'vibration_mm_s_zero_mad_deviation': 0,
  'irradiance_wm2_rolling_median': 0.0,
  'irradiance_wm2_rolling_mad': 0.0,
  'irradiance_wm2_median_deviation': 0.0,
  'irradiance_wm2_modified_zscore': 0.0,
  'irradiance_wm2_zero_mad_deviation': 0,
  'wind_speed_ms_rolling_median': 4.220000000000001,
  'wind_spe

In [22]:
import json
from pathlib import Path
from datetime import datetime

# Create output directory in project root
OUTPUT_DIR = Path("outputs") / "inference_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# 1. Save DataFrames as structured JSON
# --------------------------------------------------

def save_dataframe_json(df, filename):

    output_path = OUTPUT_DIR / filename

    df.to_json(
        output_path,
        orient="records",
        indent=2,
        date_format="iso"
    )

    print(f"Saved: {output_path}")


save_dataframe_json(
    predictions_df,
    "predictions.json"
)

save_dataframe_json(
    anomalies_df,
    "anomalies.json"
)

save_dataframe_json(
    normal_df,
    "normal.json"
)


# --------------------------------------------------
# 2. Save RAG-ready anomaly records
# --------------------------------------------------

anomaly_records_path = OUTPUT_DIR / "anomaly_records.json"

with open(
    anomaly_records_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        anomaly_records,
        file,
        indent=4,
        ensure_ascii=False,
        allow_nan=False
    )

print(f"Saved: {anomaly_records_path}")


# --------------------------------------------------
# 3. Save inference summary
# --------------------------------------------------

summary = {
    "generated_at": datetime.now().astimezone().isoformat(),

    "historical_observations": len(history_raw_df),

    "incoming_observations": len(incoming_raw_df),

    "total_predictions": len(predictions_df),

    "total_anomalies": len(anomalies_df),

    "total_normal_observations": len(normal_df),

    "total_rag_ready_records": len(anomaly_records),

    "anomalies_by_asset_type": (
        anomalies_df.groupby("asset_type")
        .size()
        .to_dict()
    )
}

summary_path = OUTPUT_DIR / "summary.json"

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        summary,
        file,
        indent=4,
        ensure_ascii=False
    )

print("\nInference results saved successfully.")
print("\nSummary:")

print(json.dumps(summary, indent=4))

Saved: outputs\inference_results\predictions.json
Saved: outputs\inference_results\anomalies.json
Saved: outputs\inference_results\normal.json
Saved: outputs\inference_results\anomaly_records.json

Inference results saved successfully.

Summary:
{
    "generated_at": "2026-10-31T03:12:30.635214+05:30",
    "historical_observations": 192,
    "incoming_observations": 48,
    "total_predictions": 48,
    "total_anomalies": 3,
    "total_normal_observations": 45,
    "total_rag_ready_records": 3,
    "anomalies_by_asset_type": {
        "wind": 3
    }
}


In [2]:
# Check sensor values for each asset type

# display(
#     raw_df.groupby("asset_id")[
#         ["irradiance_wm2", "wind_speed_ms"]
#     ].agg(["min", "max", "isna"])
# )